In [1]:
import os
import re
import hashlib
import json
import sys
from pathlib import Path
from typing import Optional
from collections import Counter
sys.path.insert(0, '../src')
from schema import Document
from cadec_loader import DRUG_ALIASES, normalize_drug_name, clean_text, load_cadec


In [2]:
doc = Document(
    text = "I took Lipitor for 3 months and my legs started aching badly.",
    source = "cadec",
    drug_name = "atorvastatin"
)
print(doc)
print("Word count: ", doc.word_count())
print("doc id: ", doc.doc_id)

# Reproducibility check, same content must give same ID
doc2 = Document(
    text="I took Lipitor for 3 months and my legs started aching badly.",
    source="cadec",
    drug_name="atorvastatin"
)
print("stable id:", doc.doc_id == doc2.doc_id) 

# Different content → different ID
doc3 = Document(
    text="Ibuprofen gave me a stomach ulcer.",
    source="cadec",
    drug_name="ibuprofen"
)
print("different id:", doc.doc_id != doc3.doc_id)  

Document(id=d0aaba0d4d9c, source='cadec', drug='atorvastatin', words=12, preview='I took Lipitor for 3 months and my legs started ac')
Word count:  12
doc id:  d0aaba0d4d9c
stable id: True
different id: True


In [3]:
SAMPLE_POSTS = [
    {
        "drug": "atorvastatin",
        "text": "Been on Lipitor for about 6 months and my legs are killing me. "
                "The muscle pain started around week 3 and keeps getting worse. "
                "My thighs ache constantly, I can barely get up the stairs. "
                "Also having memory problems which I never had before."
    },
    {
        "drug": "atorvastatin",
        "text": "Lipitor worked great for my cholesterol but destroyed my liver. "
                "My enzymes went through the roof at my 3 month check. "
                "Had to stop immediately."
    },
    {
        "drug": "ibuprofen",
        "text": "Took ibuprofen for a week for back pain and ended up in hospital "
                "with a bleeding stomach ulcer. Never had stomach problems before. "
                "Wish someone had warned me about this risk."
    },
    {
        "drug": "ibuprofen",
        "text": "My kidneys were badly affected by long term ibuprofen use. "
                "Taking 600mg three times a day for chronic pain. "
                "Creatinine went up and now I have early stage kidney damage. I am only 42."
    },
    {
        "drug": "duloxetine",
        "text": "Cymbalta was a nightmare to come off. Brain zaps every few minutes, "
                "dizziness, crying constantly. Took 4 months of slow tapering to get off properly. "
                "My doctor had no idea how bad discontinuation syndrome would be."
    },
    {
        "drug": "duloxetine",
        "text": "The nausea on Cymbalta was unbearable for the first month. "
                "Lost 4kg because I could not eat properly. "
                "Depression is better now but that first month was really rough."
    },
    {
        "drug": "diclofenac",
        "text": "Had a heart scare on diclofenac. Chest tightness and shortness of breath "
                "after 3 months of use. Cardiologist said NSAIDs increase cardiac risk "
                "especially in older patients. Stopped immediately."
    },
    {
        "drug": "diclofenac",
        "text": "Diclofenac made my tinnitus so much worse. Already had mild ringing "
                "but after 2 weeks it became loud and constant. "
                "Stopped taking it and slowly improving."
    },
    {
        "drug": "aripiprazole",
        "text": "Abilify gave me an uncontrollable urge to gamble. "
                "Never set foot in a casino and within 2 months losing hundreds weekly. "
                "Doctor never warned me about impulse control issues. "
                "This is a known side effect apparently."
    },
    {
        "drug": "aripiprazole",
        "text": "Severe akathisia on aripiprazole. Cannot sit still, "
                "constant need to move my legs. Feels like internal restlessness I cannot describe. "
                "Worse than the condition it is treating."
    },
]



In [5]:
docs = load_cadec("../data/cadec")
print("Total docs: ", {len(docs)})

#distribution
for drug, count in Counter(d.drug_name for d in docs).most_common():
    print(f"  {drug:<20} {count} docs")

# pirnt a few samples
print(f"\n---------Sample Documents--------")
for doc in docs[0:3]:
    print(f"\n{doc}")
    print(f"Full text: {doc.text}")

Loaded 1220 documents from ../data/cadec
Total docs:  {1220}
  atorvastatin         976 docs
  diclofenac           244 docs

---------Sample Documents--------

Document(id=3deaf7e06301, source='cadec', drug='diclofenac', words=102, preview='I feel a bit drowsy & have a little blurred vision')
Full text: I feel a bit drowsy & have a little blurred vision, so far no gastric problems.
I've been on Arthrotec 50 for over 10 years on and off, only taking it when I needed it.
Due to my arthritis getting progressively worse, to the point where I am in tears with the agony, gp's started me on 75 twice a day and I have to take it.
every day for the next month to see how I get on, here goes.
So far its been very good, pains almost gone, but I feel a bit weird, didn't have that when on 50.

Document(id=bea19a94a550, source='cadec', drug='diclofenac', words=33, preview='Hunger pangs. Brilliant, I have a new lease of lif')
Full text: Hunger pangs.
Brilliant, I have a new lease of life, i walk up & 